## STEP 1 / 7  -  COMPANY

In [ ]:
# =============================================================================
# STEP 1 / 7  -  COMPANY
# Register the company (keyed by symbol) in the v3 SQLite DB `companies` table.
# =============================================================================
import hashlib
import sqlite3
from pathlib import Path

# --- Config: change these for any NSE-listed symbol ---
SYMBOL = "RELIANCE"
FROM_DATE = "01-04-2026"
TO_DATE = "30-06-2026"
EVENT_TYPE = "Investor Presentation"  # this notebook selects investor presentations

# --- Paths (resolved relative to this notebook) ---
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "7_step_flow" else NOTEBOOK_DIR
DB_PATH = (REPO_ROOT / "v3" / "data" / "capital_nerve.db").resolve()
CATALOG_DIR = (REPO_ROOT / "v2" / "catalog").resolve()
DOCUMENTS_DIR = (REPO_ROOT / "v3" / "data" / "documents").resolve()
ENV_PATH = (REPO_ROOT / "v3" / ".env").resolve()

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Schema bootstrap (mirrors v3/db.py so the notebook works on a fresh DB).
SCHEMA = """
CREATE TABLE IF NOT EXISTS companies (
    id TEXT PRIMARY KEY, name TEXT NOT NULL, ticker TEXT, exchange TEXT,
    sector TEXT, industry TEXT, isin TEXT UNIQUE
);
CREATE TABLE IF NOT EXISTS events (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_type TEXT NOT NULL, event_date TEXT NOT NULL, fiscal_year INTEGER,
    fiscal_quarter INTEGER, title TEXT, source_url TEXT, document_id TEXT,
    status TEXT DEFAULT 'processed'
);
CREATE TABLE IF NOT EXISTS documents (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    source_url TEXT, storage_path TEXT NOT NULL, sha256 TEXT NOT NULL UNIQUE,
    title TEXT, document_kind TEXT, file_size INTEGER,
    status TEXT DEFAULT 'pending', error_message TEXT,
    ingested_at TEXT DEFAULT (datetime('now'))
);
CREATE TABLE IF NOT EXISTS extracted_values (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id), value_code TEXT NOT NULL,
    value_numeric REAL, value_text TEXT, unit TEXT, period_type TEXT,
    period_start TEXT, period_end TEXT, basis TEXT DEFAULT 'consolidated',
    segment TEXT, geography TEXT, source_text TEXT, source_page INTEGER,
    confidence REAL
);
CREATE TABLE IF NOT EXISTS metrics (
    id TEXT PRIMARY KEY, metric_code TEXT UNIQUE NOT NULL, name TEXT NOT NULL,
    formula TEXT, unit TEXT, description TEXT
);
CREATE TABLE IF NOT EXISTS metric_values (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT REFERENCES events(id), metric_id TEXT NOT NULL REFERENCES metrics(id),
    metric_value REAL NOT NULL, period_start TEXT, period_end TEXT, segment TEXT,
    calculation_data TEXT, calculated_at TEXT DEFAULT (datetime('now'))
);
CREATE TABLE IF NOT EXISTS signals (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT REFERENCES events(id), signal_type TEXT NOT NULL, title TEXT NOT NULL,
    description TEXT, direction TEXT, severity TEXT, confidence REAL, evidence TEXT,
    detected_at TEXT DEFAULT (datetime('now'))
);
"""


def db_connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


# Consistent company id derived from the symbol (kept the same in every step).
company_id = hashlib.sha256(f"{SYMBOL}:NSE".encode()).hexdigest()

with db_connect() as conn:
    conn.executescript(SCHEMA)
    conn.execute(
        """
        INSERT INTO companies (id, name, ticker, exchange)
        VALUES (?, ?, ?, 'NSE')
        ON CONFLICT(id) DO UPDATE SET ticker = excluded.ticker
        """,
        (company_id, SYMBOL, SYMBOL),
    )
    conn.commit()
    row = conn.execute("SELECT * FROM companies WHERE id = ?", (company_id,)).fetchone()

print(f"DB: {DB_PATH}")
print(f"Registered company {SYMBOL}  (company_id={company_id[:12]}...)")
print(dict(row))

## STEP 2 / 7  -  EVENT

In [ ]:
# =============================================================================
# STEP 2 / 7  -  EVENT
# Call the NSE corporate-announcements API for the company over the window,
# backfill company name/ISIN, and persist every announcement as a discovered event.
# =============================================================================
import requests

_API_URL = "https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi"
_HOMEPAGE = "https://www.nseindia.com"
_PAGE_URL = "https://www.nseindia.com/companies-listing/corporate-filings-announcements"

# NSE blocks generic Python user agents without a session cookie.
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": _PAGE_URL,
}

params = {
    "functionName": "getCorporateAnnouncement",
    "symbol": SYMBOL,
    "marketApiType": "equities",
    "subject": "",
    "fromDate": FROM_DATE,
    "toDate": TO_DATE,
}

# `session` is reused in Step 4 to download the PDF with the warmed cookies.
session = requests.Session()
session.headers.update(_HEADERS)
session.get(_HOMEPAGE, timeout=30)  # warm up cookies

response = session.get(_API_URL, params=params, timeout=30)
response.raise_for_status()
announcements = response.json()


def event_id_from_source(source_url: str) -> str:
    return hashlib.sha256(f"{company_id}:{source_url}".encode()).hexdigest()


# Backfill company name / ISIN from the first announcement (same company_id).
# The ISIN column is UNIQUE; an earlier (ISIN-keyed) run may already own it, so
# only claim the ISIN when it is free or already ours.
if announcements:
    first = announcements[0]
    isin = first.get("sm_isin")
    with db_connect() as conn:
        if isin:
            owner = conn.execute(
                "SELECT id FROM companies WHERE isin = ?", (isin,)
            ).fetchone()
            if owner is not None and owner["id"] != company_id:
                isin = None  # taken by a different company row -> don't reclaim
        conn.execute(
            """
            UPDATE companies
            SET name = COALESCE(?, name), isin = COALESCE(?, isin),
                industry = COALESCE(?, industry)
            WHERE id = ?
            """,
            (
                first.get("sm_name") or SYMBOL,
                isin,
                first.get("smIndustry"),
                company_id,
            ),
        )
        conn.commit()

# Persist every announcement as a discovered event.
stored = 0
with db_connect() as conn:
    for item in announcements:
        source_url = item.get("attchmntFile") or ""
        seed = source_url or f"{item.get('desc')}:{item.get('dt')}"
        ev_id = event_id_from_source(seed)
        conn.execute(
            """
            INSERT OR IGNORE INTO events (
                id, company_id, event_type, event_date, title, source_url, status
            ) VALUES (?, ?, ?, ?, ?, ?, 'discovered')
            """,
            (
                ev_id,
                company_id,
                (item.get("desc") or "(missing)").strip(),
                (item.get("sort_date") or "")[:10],
                item.get("attchmntText"),
                source_url or None,
            ),
        )
        stored += 1
    conn.commit()

by_desc: dict[str, list[dict]] = {}
for item in announcements:
    desc = (item.get("desc") or "(missing)").strip()
    by_desc.setdefault(desc, []).append(item)

print(f"{len(announcements)} announcements ({stored} stored as events) "
      f"across {len(by_desc)} desc buckets for {SYMBOL} [{FROM_DATE} -> {TO_DATE}]\n")
for desc, items in sorted(by_desc.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(f"  {len(items):>3}  {desc}")

## STEP 3 / 7  -  EVENT TYPE

In [ ]:
# =============================================================================
# STEP 3 / 7  -  EVENT TYPE
# Classify announcements into the three buckets and return the Investor Presentation.
# Keep the selected PDF in the same variables Step 4 expects.
# =============================================================================
import sys

V3_ROOT = (REPO_ROOT / "v3").resolve()
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))

from nse_fr_resolver import download_pdf, is_pdf_url, pdf_hash


def _text_blob(item: dict) -> str:
    parts = [
        item.get("desc") or "",
        item.get("attchmntText") or "",
        item.get("attchmntFile") or "",
    ]
    return " ".join(parts).lower()


def classify_announcement(item: dict) -> str:
    """Trimmed classifier covering the three event-type buckets this flow knows."""
    desc = (item.get("desc") or "").strip()
    blob = _text_blob(item)

    def has(*phrases: str) -> bool:
        return any(p in blob for p in phrases)

    # Earnings call transcript (a results-adjacent doc we recognise but skip).
    if has("transcript of the discussion", "earnings call transcript", "concall transcript"):
        return "Earnings Call Transcript"

    # Investor presentation.
    if desc == "Investor Presentation" or has("investor presentation"):
        return "Investor Presentation"

    # Financial results: the actual results filing, not an intimation/conf-call.
    intimation = has(
        "scheduled to be held", "audio recording", "will hold", "will be held", "informed the exchange regarding board meeting",
        "conference call", "prior intimation", "recording and transcript", "media release", "shareholder meeting"
    )
    if desc == "Financial Results" and not intimation:
        return "Financial Results"
    if has("financial results", "unaudited financial", "audited financial", "outcome of board meeting") and not intimation:
        return "Financial Results"

    return "Other"


# Attach classification + parsed timestamp to each announcement.
from datetime import datetime

for item in announcements:
    item["event_bucket"] = classify_announcement(item)

investor_presentations = [a for a in announcements if a["event_bucket"] == EVENT_TYPE]


def _sort_key(item: dict):
    try:
        return datetime.strptime(item.get("sort_date", ""), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return datetime.min


investor_presentations.sort(key=_sort_key, reverse=True)

if not investor_presentations:
    raise RuntimeError(
        f"No '{EVENT_TYPE}' announcements found for {SYMBOL} in {FROM_DATE} -> {TO_DATE}"
    )

presentation_pdfs = [
    a for a in investor_presentations
    if is_pdf_url((a.get("attchmntFile") or "").strip())
]
if not presentation_pdfs:
    raise RuntimeError(
        f"No PDF attachment found for '{EVENT_TYPE}' for {SYMBOL} in {FROM_DATE} -> {TO_DATE}"
    )

chosen = presentation_pdfs[0]
chosen_source_url = (chosen.get("attchmntFile") or "").strip()
print(f"Downloading investor presentation ({len(presentation_pdfs)} PDF candidate(s))...")
pdf_bytes = download_pdf(chosen_source_url, session, referer=_PAGE_URL)
digest = pdf_hash(pdf_bytes)
resolved_fr = {
    "url": chosen_source_url,
    "announcement": chosen,
    "classification": {
        "is_financial_report": False,
        "confidence": 1.0,
        "document_kind": "INVESTOR_PRESENTATION",
        "reasons": ["selected from Investor Presentation announcement bucket"],
    },
    "pdf_hash": digest,
    "pdf_bytes": pdf_bytes,
    "metadata_score": None,
    "recovery_needed": False,
}
event_id = event_id_from_source(chosen_source_url)

# Mark the chosen event row as the selected Investor Presentation.
with db_connect() as conn:
    conn.execute(
        "UPDATE events SET event_type = ?, status = 'selected' WHERE id = ?",
        (EVENT_TYPE, event_id),
    )
    conn.commit()

print(f"Found {len(investor_presentations)} Investor Presentation candidate(s):")
for a in investor_presentations:
    flag = "  <-- chosen" if a is chosen else ""
    print(f"  {a.get('sort_date')}  {a.get('attchmntFile', '')}{flag}")

cls = resolved_fr.get("classification") or {}
print("\nChosen Investor Presentation:")
print(f"  date : {chosen.get('sort_date')}")
print(f"  title: {chosen.get('attchmntText')}")
print(f"  pdf  : {chosen_source_url}")
print(f"  pdf classification: kind={cls.get('document_kind')} "
      f"conf={cls.get('confidence')}")
print(f"  event_id={event_id[:12]}...")

## STEP 4 / 7  -  VALUES

In [ ]:
# =============================================================================
# STEP 4 / 7  -  VALUES
# Download the chosen Investor Presentation PDF, store the document, parse to markdown,
# extract presentation-specific facts from `presentation_facts.json`, and persist
# them into `extracted_values`.
# =============================================================================
import json
import re
import sys
from datetime import date, datetime, timedelta

# ---- 4a. Load OPENAI_API_KEY from v3/.env -----------------------------------
def _load_env_value(key: str) -> str:
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith(f"{key}="):
            return line.split("=", 1)[1].strip()
    return ""


OPENAI_API_KEY = _load_env_value("OPENAI_API_KEY")
OPENAI_MODEL = _load_env_value("OPENAI_MODEL") or "gpt-4.1-mini"
OPENAI_PARSE_MODEL = _load_env_value("OPENAI_PARSE_MODEL") or OPENAI_MODEL
if not OPENAI_API_KEY:
    raise RuntimeError(f"OPENAI_API_KEY not found in {ENV_PATH}")

PARSED_DIR = (REPO_ROOT / "v3" / "data" / "parsed").resolve()
PARSED_DIR.mkdir(parents=True, exist_ok=True)

# ---- 4b. Use resolved PDF from Step 3 (already downloaded + verified) ----------
pdf_url = resolved_fr["url"]
pdf_bytes = resolved_fr["pdf_bytes"]
document_id = hashlib.sha256(pdf_bytes).hexdigest()
storage_path = DOCUMENTS_DIR / f"{document_id}.pdf"
if not storage_path.exists():
    storage_path.write_bytes(pdf_bytes)
document_kind = (resolved_fr.get("classification") or {}).get("document_kind") or "INVESTOR_PRESENTATION"

with db_connect() as conn:
    conn.execute(
        """
        INSERT OR IGNORE INTO documents (
            id, company_id, source_url, storage_path, sha256, title,
            document_kind, file_size, status
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'ingested')
        """,
        (document_id, company_id, pdf_url, str(storage_path), document_id,
         chosen.get("attchmntText"), document_kind, len(pdf_bytes)),
    )
    conn.execute(
        "UPDATE events SET document_id = ? WHERE id = ?", (document_id, event_id)
    )
    conn.commit()

# ---- 4c. Parse PDF -> markdown (LLM vision, cached under v3/data/parsed) ------
V2_ROOT = (REPO_ROOT / "v2").resolve()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import importlib

import pdf_parse
from periods import (
    detect_reporting_period,
    fy_start_year_from_date,
    quarter_end_date,
    quarter_from_date,
    reporting_period_from_date,
)

# Pick up edits to v2 modules without restarting the kernel.
importlib.reload(pdf_parse)

from openai import OpenAI
from pdf_parse import parse_pdf_to_markdown

client = OpenAI(api_key=OPENAI_API_KEY)
print("Parsing investor presentation PDF to markdown (uses cache when available)...")
pdf_markdown = parse_pdf_to_markdown(
    storage_path,
    parsed_dir=PARSED_DIR,
    client=client,
    model=OPENAI_PARSE_MODEL,
)
print(f"Markdown length: {len(pdf_markdown):,} chars")

# ---- 4d. Detect reporting period ----------------------------------------------
reporting_period = detect_reporting_period(
    pdf_markdown, title=chosen.get("attchmntText") or ""
)
if reporting_period is None:
    ann = datetime.strptime(chosen["sort_date"], "%Y-%m-%d %H:%M:%S").date()
    fy = fy_start_year_from_date(ann)
    q = quarter_from_date(ann)
    reporting_period = reporting_period_from_date(
        quarter_end_date(q, fy), "announcement_fallback"
    )

period_date = date.fromisoformat(reporting_period.quarter_end)
PERIOD_QUARTER = reporting_period.quarter
PERIOD_FY_START = reporting_period.fy_start_year
PERIOD_END = reporting_period.quarter_end
PERIOD_LABEL = reporting_period.label
print(f"Reporting period: {PERIOD_LABEL}  (quarter_end={PERIOD_END})")

# ---- 4e. Presentation fact catalog + LLM extraction --------------------------
PRESENTATION_CATALOG_DIR = (
    REPO_ROOT / "v4" / "microservices" / "catalog" / "investor_presentation"
).resolve()
facts_catalog = json.loads(
    (PRESENTATION_CATALOG_DIR / "presentation_facts.json").read_text(encoding="utf-8")
)

storage_to_fact: dict[str, str] = {}
fact_lines: list[str] = []
for key, spec in facts_catalog.items():
    storage_to_fact[key] = key
    for alias in spec.get("aliases") or []:
        storage_to_fact[str(alias)] = key
    aliases = ", ".join(spec.get("aliases") or [])
    dimensions = ", ".join(spec.get("dimensions") or [])
    allowed = ", ".join(spec.get("allowed_values") or [])
    fact_type = spec.get("fact_type") or "NUMERIC"
    notes = []
    if aliases:
        notes.append(f"aliases: {aliases}")
    if dimensions:
        notes.append(f"dimensions: {dimensions}")
    if allowed:
        notes.append(f"allowed_values: {allowed}")
    note = f" ({'; '.join(notes)})" if notes else ""
    fact_lines.append(
        f"- {key}: {spec.get('name')} [{spec.get('unit')}; {fact_type}; {spec.get('statement')}]{note}"
    )
fact_catalog_text = "\n".join(fact_lines)


def _chunk(text: str, max_chars: int = 14000) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        if end < len(text):
            brk = text.rfind("\n\n", start, end)
            if brk > start:
                end = brk
        chunks.append(text[start:end])
        start = end
    return chunks


def extract_facts_from_chunk(chunk: str) -> list[dict]:
    prompt = f"""Extract investor-presentation facts from this Indian company presentation markdown.

Document type: INVESTOR_PRESENTATION
Company: {SYMBOL}
Reporting period context: {PERIOD_LABEL} (quarter_end={PERIOD_END})

Allowed fact_key values (use only canonical keys from this catalog):
{fact_catalog_text}

Extraction rules:
- Extract ONLY values explicitly stated in the markdown. Do not calculate, infer, annualize, or fill missing values.
- This is a presentation, not a statutory financial-results table. Look for slide KPIs, charts, segment/product/geography tables, order book/inflow, capacity, utilization, capex, projects, debt/cash, guidance, and management outlook.
- Prefer values for the current/latest reported period. Also extract forward-looking guidance, planned capacity additions, and expected commissioning dates when the slide explicitly labels them.
- Preserve business dimensions when present: segment, product, geography, project, plant, fiscal_year, and period_label.
- For NUMERIC facts, set numeric_value to a number with commas stripped and text_value to null.
- For DATE facts, set text_value to the exact date/quarter/month label and numeric_value to null.
- For CATEGORICAL facts, set text_value to one allowed catalog value exactly and numeric_value to null.
- unit should be the exact reported unit when possible; use catalog units like crore, %, count, date, enum, company_reported, or currency_per_unit when the slide unit is generic.
- basis should be "presentation" unless the slide explicitly says consolidated or standalone.
- evidence must be a short snippet containing the value and its label.
- confidence must be between 0.0 and 1.0.

Return JSON object only:
{{"facts": [{{"fact_key": "...", "numeric_value": 0.0, "text_value": null, "unit": "...", "basis": "presentation", "segment": null, "product": null, "geography": null, "project": null, "plant": null, "fiscal_year": null, "period_label": "...", "evidence": "...", "confidence": 0.9}}]}}
If no catalog facts are present, return {{"facts": []}}.

Markdown:
{chunk}
"""
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[{"role": "user", "content": prompt}],
        text={"format": {"type": "json_object"}},
        temperature=0,
    )
    payload = json.loads((response.output_text or "{}").strip())
    facts = payload.get("facts") or []
    return [f for f in facts if isinstance(f, dict)]


raw_facts: list[dict] = []
for chunk in _chunk(pdf_markdown):
    raw_facts.extend(extract_facts_from_chunk(chunk))
print(f"LLM presentation extraction: {len(raw_facts)} raw fact(s)")

# ---- 4f. Validate, canonicalize, and preserve presentation dimensions --------
def _canon_unit(unit):
    if not unit:
        return None
    u = str(unit).strip()
    lower = u.lower()
    return {
        "crores": "crore", "cr": "crore", "rs.": "Rs", "rs": "Rs",
        "percent": "%", "pct": "%", "percentage": "%",
    }.get(lower, u)


def _clean_str(value):
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def _dimension_payload(entry: dict) -> dict[str, str]:
    dims: dict[str, str] = {}
    for key in ("segment", "product", "geography", "project", "plant", "fiscal_year", "period_label"):
        value = _clean_str(entry.get(key))
        if value:
            dims[key] = value
    return dims


def _primary_segment(dims: dict[str, str]) -> str | None:
    return dims.get("segment") or dims.get("product") or dims.get("project") or dims.get("plant") or dims.get("fiscal_year")


def _source_text(evidence: str, dims: dict[str, str]) -> str:
    if not dims:
        return evidence
    dim_text = ", ".join(f"{k}={v}" for k, v in dims.items())
    return f"{evidence} | dimensions: {dim_text}" if evidence else f"dimensions: {dim_text}"


cleaned: dict[tuple, dict] = {}
for entry in raw_facts:
    fk = entry.get("fact_key")
    if not fk:
        continue
    canonical = storage_to_fact.get(str(fk), str(fk))
    spec = facts_catalog.get(canonical)
    if spec is None:
        continue

    fact_type = str(spec.get("fact_type") or "NUMERIC").upper()
    numeric_value = None
    raw_value = entry.get("numeric_value", entry.get("value"))
    value_text = _clean_str(entry.get("text_value") or entry.get("value_text") or entry.get("value"))
    if fact_type == "NUMERIC":
        try:
            numeric_value = float(str(raw_value).replace(",", ""))
        except (TypeError, ValueError):
            continue
    elif fact_type == "DATE":
        if not value_text:
            value_text = _clean_str(entry.get("numeric_value"))
        if not value_text:
            continue
    elif fact_type == "CATEGORICAL":
        if not value_text:
            continue
        allowed = {str(v).upper() for v in spec.get("allowed_values") or []}
        value_text = value_text.strip().upper()
        if allowed and value_text not in allowed:
            continue
    else:
        if value_text is None and entry.get("numeric_value") is not None:
            value_text = _clean_str(entry.get("numeric_value"))
        if value_text is None:
            continue

    basis = (entry.get("basis") or "presentation").strip().lower()
    conf = float(entry.get("confidence") or 0.7)
    dims = _dimension_payload(entry)
    evidence = _clean_str(entry.get("evidence")) or ""
    row = {
        "fact_key": canonical,
        "numeric_value": numeric_value,
        "value_text": value_text,
        "unit": _canon_unit(entry.get("unit")) or spec.get("unit"),
        "basis": basis,
        "segment": _primary_segment(dims),
        "geography": dims.get("geography"),
        "dimensions": dims,
        "period_label": dims.get("period_label"),
        "evidence": evidence,
        "source_text": _source_text(evidence, dims),
        "confidence": conf,
    }
    dedupe_key = (
        canonical,
        basis,
        row["segment"],
        row["geography"],
        row["period_label"],
        value_text if fact_type != "NUMERIC" else None,
    )
    if dedupe_key not in cleaned or conf > cleaned[dedupe_key]["confidence"]:
        cleaned[dedupe_key] = row

accepted_rows = list(cleaned.values())
if not accepted_rows:
    raise RuntimeError("No presentation facts passed validation after extraction")

# ---- 4g. Persist into extracted_values ---------------------------------------
def value_id(row: dict) -> str:
    parts = [
        company_id,
        row["fact_key"],
        PERIOD_END,
        row["basis"],
        row.get("segment") or "",
        row.get("geography") or "",
        row.get("period_label") or "",
        row.get("value_text") or "",
    ]
    return hashlib.sha256(":".join(parts).encode()).hexdigest()


with db_connect() as conn:
    for row in accepted_rows:
        vid = value_id(row)
        conn.execute(
            """
            INSERT INTO extracted_values (
                id, company_id, event_id, value_code, value_numeric, value_text, unit,
                period_type, period_start, period_end, basis, segment, geography,
                source_text, confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 'quarter', NULL, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                event_id = excluded.event_id,
                value_numeric = excluded.value_numeric,
                value_text = excluded.value_text,
                unit = excluded.unit,
                basis = excluded.basis,
                segment = excluded.segment,
                geography = excluded.geography,
                source_text = excluded.source_text,
                confidence = excluded.confidence
            """,
            (vid, company_id, event_id, row["fact_key"], row["numeric_value"],
             row["value_text"], row["unit"], PERIOD_END, row["basis"],
             row["segment"], row["geography"], row["source_text"], row["confidence"]),
        )
    conn.execute(
        "UPDATE events SET status = 'processed', fiscal_year = ?, fiscal_quarter = ? WHERE id = ?",
        (PERIOD_FY_START, PERIOD_QUARTER, event_id),
    )
    conn.execute(
        "UPDATE documents SET status = 'processed' WHERE id = ?", (document_id,)
    )
    conn.commit()

print(f"\nExtracted {len(accepted_rows)} presentation fact(s) for {PERIOD_LABEL}:")
for row in sorted(accepted_rows, key=lambda r: (r["fact_key"], r.get("segment") or "", r.get("geography") or "")):
    value = row["numeric_value"] if row["numeric_value"] is not None else row["value_text"]
    if isinstance(value, float):
        value_str = f"{value:,.2f}"
    else:
        value_str = str(value)
    dim_bits = []
    if row.get("segment"):
        dim_bits.append(f"segment={row['segment']}")
    if row.get("geography"):
        dim_bits.append(f"geography={row['geography']}")
    dim_note = f" ({', '.join(dim_bits)})" if dim_bits else ""
    print(f"  {row['fact_key']:<32} {value_str:>14} {row['unit'] or '':<16} [{row['basis']}]{dim_note}")


## STEP 5 / 7  -  PRESENTATION METRICS

In [14]:
# =============================================================================
# STEP 5 / 7  -  PRESENTATION METRICS
# Seed the investor-presentation metric catalog, load presentation facts across
# relevant scopes, evaluate catalog formulas, and persist into `metric_values`.
# =============================================================================
import math
from datetime import date, datetime

PRESENTATION_CATALOG_DIR = globals().get("PRESENTATION_CATALOG_DIR") or (
    REPO_ROOT / "v4" / "microservices" / "catalog" / "investor_presentation"
).resolve()
metrics_catalog = json.loads(
    (PRESENTATION_CATALOG_DIR / "presentation_metrics.json").read_text(encoding="utf-8")
)

# ---- 5a. Seed presentation metric catalog -----------------------------------
with db_connect() as conn:
    for code, spec in metrics_catalog.items():
        mid = hashlib.sha256(code.encode()).hexdigest()
        formula_payload = json.dumps({
            "formula": spec.get("formula"),
            "formula_type": spec.get("formula_type"),
            "inputs": spec.get("inputs") or [],
            "category": spec.get("category"),
            "preserve_dimensions": bool(spec.get("preserve_dimensions")),
        })
        conn.execute(
            """
            INSERT INTO metrics (id, metric_code, name, formula, unit, description)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(metric_code) DO UPDATE SET
                name = excluded.name, formula = excluded.formula,
                unit = excluded.unit, description = excluded.description
            """,
            (mid, code, spec.get("name", code), formula_payload,
             spec.get("unit"), spec.get("category")),
        )
    conn.commit()

# ---- 5b. Load presentation fact rows by scope --------------------------------
def prior_year_end() -> str:
    return quarter_end_date(PERIOD_QUARTER, PERIOD_FY_START - 1).isoformat()


def prior_quarter_end() -> str:
    if PERIOD_QUARTER == 1:
        q, fy = 4, PERIOD_FY_START - 1
    else:
        q, fy = PERIOD_QUARTER - 1, PERIOD_FY_START
    return quarter_end_date(q, fy).isoformat()


def _dimension_key(row: dict) -> tuple[str, str]:
    return (row.get("segment") or "", row.get("geography") or "")


def _row_from_db(r) -> dict:
    return {
        "fact_key": r["value_code"],
        "value": float(r["value_numeric"]) if r["value_numeric"] is not None else r["value_text"],
        "numeric_value": float(r["value_numeric"]) if r["value_numeric"] is not None else None,
        "value_text": r["value_text"],
        "unit": r["unit"],
        "basis": r["basis"],
        "segment": r["segment"],
        "geography": r["geography"],
        "period_end": r["period_end"],
        "source_text": r["source_text"],
        "confidence": float(r["confidence"] or 0),
    }


def _rows_to_pool(rows: list[dict]) -> dict[str, list[dict]]:
    pool: dict[str, list[dict]] = {}
    for row in rows:
        pool.setdefault(row["fact_key"], []).append(row)
    for fact_rows in pool.values():
        fact_rows.sort(key=lambda r: (r.get("period_end") or "", r.get("confidence") or 0), reverse=True)
    return pool


def load_fact_rows(scope: str) -> dict[str, list[dict]]:
    scope = scope.upper()
    with db_connect() as conn:
        if scope in {"CURRENT", "CURRENT_DISCLOSURE", "CUMULATIVE", "YTD"}:
            rows = conn.execute(
                """
                SELECT value_code, value_numeric, value_text, unit, basis, segment,
                       geography, period_end, source_text, confidence
                FROM extracted_values
                WHERE company_id = ? AND event_id = ? AND period_end = ?
                ORDER BY confidence DESC
                """,
                (company_id, event_id, PERIOD_END),
            ).fetchall()
        elif scope == "PY":
            rows = conn.execute(
                """
                SELECT value_code, value_numeric, value_text, unit, basis, segment,
                       geography, period_end, source_text, confidence
                FROM extracted_values
                WHERE company_id = ? AND period_end = ?
                ORDER BY confidence DESC
                """,
                (company_id, prior_year_end()),
            ).fetchall()
        elif scope == "PQ":
            rows = conn.execute(
                """
                SELECT value_code, value_numeric, value_text, unit, basis, segment,
                       geography, period_end, source_text, confidence
                FROM extracted_values
                WHERE company_id = ? AND period_end = ?
                ORDER BY confidence DESC
                """,
                (company_id, prior_quarter_end()),
            ).fetchall()
        elif scope == "PREVIOUS_DISCLOSURE":
            rows = conn.execute(
                """
                SELECT value_code, value_numeric, value_text, unit, basis, segment,
                       geography, period_end, source_text, confidence
                FROM extracted_values
                WHERE company_id = ? AND period_end < ?
                ORDER BY period_end DESC, confidence DESC
                """,
                (company_id, PERIOD_END),
            ).fetchall()
        else:
            rows = []
    return _rows_to_pool([_row_from_db(r) for r in rows])


_SCOPE_NAMES = {"CURRENT", "PY", "PQ", "CURRENT_DISCLOSURE", "PREVIOUS_DISCLOSURE", "CUMULATIVE", "YTD"}
SCOPE_POOLS = {scope: load_fact_rows(scope) for scope in _SCOPE_NAMES}
facts_current = {k: rows[0] for k, rows in SCOPE_POOLS["CURRENT"].items() if rows}
facts_py = {k: rows[0] for k, rows in SCOPE_POOLS["PY"].items() if rows}
facts_pq = {k: rows[0] for k, rows in SCOPE_POOLS["PQ"].items() if rows}

# ---- 5c. Formula helpers -----------------------------------------------------
_UNIT_SCALE = {
    "crore": 1.0, "crores": 1.0, "cr": 1.0, "lakh": 0.01, "lakhs": 0.01,
    "lac": 0.01, "lacs": 0.01, "million": 0.1, "mn": 0.1, "billion": 100.0,
    "bn": 100.0, "thousand": 1e-5,
}
_PASS_THROUGH = {"%", "percent", "pct", "bps", "pp", "rs", "rs.", "inr", "x", "times", "days", "count"}


def crore_scale(unit):
    if not unit:
        return None
    u = str(unit).strip().lower()
    if u in _PASS_THROUGH or "per" in u or u == "company_reported":
        return None
    return _UNIT_SCALE.get(u)


def _candidate_rows(inp: dict, dim_key: tuple[str, str] | None = None) -> list[dict]:
    scope = inp.get("scope", "CURRENT").upper()
    rows = SCOPE_POOLS.get(scope, {}).get(inp["fact_key"], [])
    rows = [r for r in rows if r.get("numeric_value") is not None or inp.get("allow_text")]
    if not dim_key:
        return rows
    seg, geo = dim_key
    exact = [r for r in rows if _dimension_key(r) == dim_key]
    if exact:
        return exact
    seg_only = [r for r in rows if seg and (r.get("segment") or "") == seg]
    if seg_only:
        return seg_only
    geo_only = [r for r in rows if geo and (r.get("geography") or "") == geo]
    if geo_only:
        return geo_only
    company_level = [r for r in rows if _dimension_key(r) == ("", "")]
    return company_level or rows


def _numeric_value(row: dict, comparable: bool) -> float | None:
    val = row.get("numeric_value")
    if val is None:
        return None
    if comparable:
        scale = crore_scale(row.get("unit"))
        if scale is not None:
            val *= scale
    return float(val)


def _parse_dateish(value) -> date | None:
    if value is None:
        return None
    text = str(value).strip()
    for fmt in ("%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%d %B %Y", "%d %b %Y", "%B %Y", "%b %Y"):
        try:
            parsed = datetime.strptime(text, fmt)
            return parsed.date()
        except ValueError:
            pass
    match = re.search(r"(20\d{2})", text)
    if match:
        return date(int(match.group(1)), 3, 31)
    return None


def date_diff_days(current_value, previous_value) -> float | None:
    current = _parse_dateish(current_value)
    previous = _parse_dateish(previous_value)
    if current is None or previous is None:
        return None
    return float((current - previous).days)


def _fact_scopes(inputs: list[dict]) -> set[str]:
    return {inp.get("scope", "CURRENT").upper() for inp in inputs if "fact_key" in inp}


def resolve_inputs(inputs: list[dict], *, dim_key: tuple[str, str] | None = None, formula_type: str = "FORMULA"):
    comparable = len(_fact_scopes(inputs)) > 1
    resolved: dict[str, object] = {}
    for inp in inputs:
        if "constant" in inp:
            resolved[inp["var"]] = inp["constant"]
            continue
        if "runtime_parameter" in inp:
            if inp["runtime_parameter"] == "annualization_factor":
                resolved[inp["var"]] = 4.0 / PERIOD_QUARTER
                continue
            return None
        if "fact_key" not in inp:
            return None
        rows = _candidate_rows(inp, dim_key=dim_key)
        if not rows:
            if inp.get("optional"):
                resolved[inp["var"]] = 0.0
                continue
            return None
        if formula_type == "AGGREGATION" and inp.get("var") != "revenue":
            values = [_numeric_value(r, comparable) for r in rows]
            values = [v for v in values if v is not None]
            if not values:
                return None
            resolved[inp["var"]] = values
        elif formula_type == "DATE":
            resolved[inp["var"]] = rows[0].get("value_text") or rows[0].get("value")
        else:
            val = _numeric_value(rows[0], comparable)
            if val is None:
                return None
            resolved[inp["var"]] = val
    return resolved


def safe_eval(formula: str, variables: dict[str, object], *, formula_type: str = "FORMULA"):
    namespace = {
        "__builtins__": {},
        "min": min,
        "max": max,
        "abs": abs,
        "sum": sum,
        "math": math,
        "average": lambda a, b: (a + b) / 2,
        "sum_by_dimension": lambda values: sum(values) if isinstance(values, list) else values,
        "max_by_dimension": lambda values: max(values) if isinstance(values, list) and values else values,
        "date_diff_days": date_diff_days,
        **variables,
    }
    try:
        result = eval(formula, namespace)  # trusted catalog formulas only
    except (ZeroDivisionError, TypeError, NameError, ValueError):
        return None
    if result is None:
        return None
    return float(result) if isinstance(result, (int, float)) else None


def _input_labels(inputs: list[dict]) -> list[str]:
    return [inp.get("fact_key") or inp.get("metric_key") or inp.get("runtime_parameter") or "constant" for inp in inputs]


def _candidate_dimension_keys(spec: dict) -> list[tuple[str, str] | None]:
    if not spec.get("preserve_dimensions"):
        return [None]
    keys: set[tuple[str, str]] = set()
    for inp in spec.get("inputs") or []:
        if "fact_key" not in inp:
            continue
        scope = inp.get("scope", "CURRENT").upper()
        if scope not in {"CURRENT", "CURRENT_DISCLOSURE", "CUMULATIVE", "YTD"}:
            continue
        for row in SCOPE_POOLS.get(scope, {}).get(inp["fact_key"], []):
            key = _dimension_key(row)
            if key != ("", ""):
                keys.add(key)
    return sorted(keys) if keys else [None]


# ---- 5d. Compute and persist presentation metric values ----------------------
computed_metrics: list[dict] = []
for code, spec in metrics_catalog.items():
    inputs_spec = spec.get("inputs") or []
    formula_type = spec.get("formula_type") or "FORMULA"
    for dim_key in _candidate_dimension_keys(spec):
        variables = resolve_inputs(inputs_spec, dim_key=dim_key, formula_type=formula_type)
        if variables is None:
            continue
        value = safe_eval(spec["formula"], variables, formula_type=formula_type)
        if value is None:
            continue
        segment, geography = dim_key or ("", "")
        computed_metrics.append({
            "metric_key": code,
            "name": spec.get("name", code),
            "value": round(value, 2),
            "unit": spec.get("unit"),
            "category": spec.get("category"),
            "formula": spec["formula"],
            "formula_type": formula_type,
            "inputs": sorted(set(_input_labels(inputs_spec))),
            "segment": segment or None,
            "geography": geography or None,
        })

with db_connect() as conn:
    conn.execute(
        "DELETE FROM metric_values WHERE company_id = ? AND event_id = ?",
        (company_id, event_id),
    )
    for m in computed_metrics:
        mid_row = conn.execute(
            "SELECT id FROM metrics WHERE metric_code = ?", (m["metric_key"],)
        ).fetchone()
        if not mid_row:
            continue
        mvid = hashlib.sha256(
            f"{company_id}:{event_id}:{m['metric_key']}:{m.get('segment') or ''}:{m.get('geography') or ''}".encode()
        ).hexdigest()
        conn.execute(
            """
            INSERT INTO metric_values (
                id, company_id, event_id, metric_id, metric_value,
                period_start, period_end, segment, calculation_data
            ) VALUES (?, ?, ?, ?, ?, NULL, ?, ?, ?)
            """,
            (mvid, company_id, event_id, mid_row["id"], m["value"], PERIOD_END,
             m.get("segment"),
             json.dumps({
                 "unit": m["unit"], "formula": m["formula"],
                 "formula_type": m["formula_type"], "inputs": m["inputs"],
                 "segment": m.get("segment"), "geography": m.get("geography"),
                 "catalog": "investor_presentation/presentation_metrics.json",
             })),
        )
    conn.commit()

print(f"Computed {len(computed_metrics)} presentation metric value(s) for {PERIOD_LABEL}")
if not facts_py:
    print("  (no prior-year presentation facts in DB -> YoY metrics skipped)")
if not facts_pq:
    print("  (no prior-quarter presentation facts in DB -> QoQ metrics skipped)")
print()
for m in computed_metrics:
    dims = []
    if m.get("segment"):
        dims.append(f"segment={m['segment']}")
    if m.get("geography"):
        dims.append(f"geography={m['geography']}")
    dim_note = f" [{', '.join(dims)}]" if dims else ""
    print(f"  {m['name']:<38} {m['value']:>12,.2f} {m['unit'] or '':<8} [{m['category']}]{dim_note}")


Computed 20 presentation metric value(s) for Q4 FY2025-26
  (no prior-quarter presentation facts in DB -> QoQ metrics skipped)

  Production to Sales Ratio                      8.16 x        [inventory_proxy] [segment=ATF, geography=Domestic India]
  Production to Sales Ratio                      0.00 x        [inventory_proxy] [segment=Connectivity Business]
  Production to Sales Ratio                      0.80 x        [inventory_proxy] [segment=Diesel, geography=Domestic India]
  Production to Sales Ratio                      0.00 x        [inventory_proxy] [segment=Digital Services, geography=India]
  Production to Sales Ratio                      0.00 x        [inventory_proxy] [segment=Downstream]
  Production to Sales Ratio                      1.82 x        [inventory_proxy] [segment=Gasoline, geography=Domestic India]
  Production to Sales Ratio                      0.00 x        [inventory_proxy] [segment=Oil and Gas]
  Segment EBITDA Margin                         52.12 %   

## STEP 6 / 7  -  PRESENTATION SIGNALS

In [15]:
# =============================================================================
# STEP 6 / 7  -  PRESENTATION SIGNALS
# Evaluate investor-presentation signal rules against computed presentation
# metrics/facts and persist every fired signal into `signals`.
# =============================================================================
PRESENTATION_CATALOG_DIR = globals().get("PRESENTATION_CATALOG_DIR") or (
    REPO_ROOT / "v4" / "microservices" / "catalog" / "investor_presentation"
).resolve()
signals_catalog = json.loads(
    (PRESENTATION_CATALOG_DIR / "presentation_signals.json").read_text(encoding="utf-8")
)


def _metric_dim_key(row: dict) -> tuple[str, str]:
    return (row.get("segment") or "", row.get("geography") or "")


metrics_by_key: dict[str, list[dict]] = {}
for m in computed_metrics:
    metrics_by_key.setdefault(m["metric_key"], []).append(m)

facts_by_key: dict[str, list[dict]] = {}
for rows in SCOPE_POOLS.get("CURRENT", {}).values():
    for row in rows:
        facts_by_key.setdefault(row["fact_key"], []).append(row)
for row in globals().get("accepted_rows", []):
    if "fact_key" not in row:
        continue
    value = row.get("numeric_value") if row.get("numeric_value") is not None else row.get("value_text")
    facts_by_key.setdefault(row["fact_key"], []).append({
        "fact_key": row["fact_key"],
        "value": value,
        "unit": row.get("unit"),
        "segment": row.get("segment"),
        "geography": row.get("geography"),
    })

_OPS = {
    ">": lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<": lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
    "==": lambda a, b: a == b,
    "!=": lambda a, b: a != b,
    "IN": lambda a, b: isinstance(b, (list, tuple, set)) and a in b,
    "NOT IN": lambda a, b: isinstance(b, (list, tuple, set)) and a not in b,
    "ABS_GT": lambda a, b: abs(a) > b,
}


def _refs_for_key(pool: dict[str, list[dict]], key: str, dim_key: tuple[str, str] | None) -> list[dict]:
    rows = pool.get(key, [])
    if not dim_key:
        company = [r for r in rows if _metric_dim_key(r) == ("", "")]
        return company or rows
    exact = [r for r in rows if _metric_dim_key(r) == dim_key]
    if exact:
        return exact
    seg, geo = dim_key
    seg_only = [r for r in rows if seg and (r.get("segment") or "") == seg]
    if seg_only:
        return seg_only
    geo_only = [r for r in rows if geo and (r.get("geography") or "") == geo]
    if geo_only:
        return geo_only
    company = [r for r in rows if _metric_dim_key(r) == ("", "")]
    return company or rows


def _ref_value(row: dict):
    if "value" in row:
        return row["value"]
    if "numeric_value" in row and row["numeric_value"] is not None:
        return row["numeric_value"]
    return row.get("value_text")


def eval_leaf(rule: dict, dim_key: tuple[str, str] | None = None) -> bool:
    if "metric_key" in rule:
        refs = _refs_for_key(metrics_by_key, rule["metric_key"], dim_key)
        value_key = rule["metric_key"]
    elif "fact_key" in rule:
        refs = _refs_for_key(facts_by_key, rule["fact_key"], dim_key)
        value_key = rule["fact_key"]
    else:
        raise ValueError(f"Malformed rule leaf: {rule}")
    if not refs:
        return False
    left = _ref_value(refs[0])

    if "compare_metric_key" in rule:
        compare_refs = _refs_for_key(metrics_by_key, rule["compare_metric_key"], dim_key)
        if not compare_refs:
            return False
        right = _ref_value(compare_refs[0])
    elif "compare_fact_key" in rule:
        compare_refs = _refs_for_key(facts_by_key, rule["compare_fact_key"], dim_key)
        if not compare_refs:
            return False
        right = _ref_value(compare_refs[0])
    else:
        right = rule.get("value")

    op = rule["op"].upper() if isinstance(rule["op"], str) else rule["op"]
    if op not in _OPS:
        raise ValueError(f"Unsupported rule operator: {rule['op']}")
    try:
        return bool(_OPS[op](left, right))
    except TypeError:
        return False


def eval_rule(rule: dict, dim_key: tuple[str, str] | None = None) -> bool:
    if "all" in rule:
        return all(eval_rule(child, dim_key=dim_key) for child in rule["all"])
    if "any" in rule:
        return any(eval_rule(child, dim_key=dim_key) for child in rule["any"])
    if "metric_key" in rule or "fact_key" in rule:
        return eval_leaf(rule, dim_key=dim_key)
    raise ValueError(f"Malformed rule: {rule}")


def rule_metric_keys(rule: dict) -> list[str]:
    keys: list[str] = []
    if "metric_key" in rule:
        keys.append(rule["metric_key"])
    if "compare_metric_key" in rule:
        keys.append(rule["compare_metric_key"])
    for k in ("all", "any"):
        for child in rule.get(k, []):
            keys.extend(rule_metric_keys(child))
    return keys


def rule_fact_keys(rule: dict) -> list[str]:
    keys: list[str] = []
    if "fact_key" in rule:
        keys.append(rule["fact_key"])
    if "compare_fact_key" in rule:
        keys.append(rule["compare_fact_key"])
    for k in ("all", "any"):
        for child in rule.get(k, []):
            keys.extend(rule_fact_keys(child))
    return keys


def format_rule(rule: dict) -> str:
    if "metric_key" in rule:
        rhs = rule.get("compare_metric_key", rule.get("value"))
        return f"{rule['metric_key']} {rule['op']} {rhs}"
    if "fact_key" in rule:
        rhs = rule.get("compare_fact_key", rule.get("value"))
        return f"{rule['fact_key']} {rule['op']} {rhs}"
    if "all" in rule:
        return " AND ".join(f"({format_rule(c)})" for c in rule["all"])
    if "any" in rule:
        return " OR ".join(f"({format_rule(c)})" for c in rule["any"])
    return ""


def _candidate_signal_dims(metric_keys: list[str], fact_keys: list[str]) -> list[tuple[str, str] | None]:
    keys: set[tuple[str, str]] = set()
    for key in metric_keys:
        for row in metrics_by_key.get(key, []):
            dim = _metric_dim_key(row)
            if dim != ("", ""):
                keys.add(dim)
    for key in fact_keys:
        for row in facts_by_key.get(key, []):
            dim = _metric_dim_key(row)
            if dim != ("", ""):
                keys.add(dim)
    return sorted(keys) + [None] if keys else [None]


def _trigger_values(metric_keys: list[str], fact_keys: list[str], dim_key: tuple[str, str] | None) -> dict:
    trigger = {}
    for key in metric_keys:
        refs = _refs_for_key(metrics_by_key, key, dim_key)
        if refs:
            trigger[key] = _ref_value(refs[0])
    for key in fact_keys:
        refs = _refs_for_key(facts_by_key, key, dim_key)
        if refs:
            trigger[key] = _ref_value(refs[0])
    return trigger


fired_signals: list[dict] = []
for code, spec in signals_catalog.items():
    rule = spec.get("rule")
    if not rule:
        continue
    metric_keys = sorted(set(rule_metric_keys(rule)))
    fact_keys = sorted(set(rule_fact_keys(rule)))
    if not all(k in metrics_by_key for k in metric_keys):
        continue
    if not all(k in facts_by_key for k in fact_keys):
        continue

    fired_dim = None
    for dim_key in _candidate_signal_dims(metric_keys, fact_keys):
        if eval_rule(rule, dim_key=dim_key):
            fired_dim = dim_key
            break
    if fired_dim is None:
        continue

    trigger = _trigger_values(metric_keys, fact_keys, fired_dim)
    segment, geography = fired_dim or ("", "")
    fired_signals.append({
        "signal_key": code,
        "title": spec.get("name", code),
        "description": spec.get("description", ""),
        "direction": spec.get("direction"),
        "severity": spec.get("severity"),
        "category": spec.get("category"),
        "metric_keys": metric_keys,
        "fact_keys": fact_keys,
        "trigger_values": trigger,
        "rule_text": format_rule(rule),
        "segment": segment or None,
        "geography": geography or None,
    })

# ---- Persist fired presentation signals -------------------------------------
with db_connect() as conn:
    conn.execute(
        "DELETE FROM signals WHERE company_id = ? AND event_id = ?",
        (company_id, event_id),
    )
    for s in fired_signals:
        sid = hashlib.sha256(
            f"{company_id}:{event_id}:{s['signal_key']}:{s.get('segment') or ''}:{s.get('geography') or ''}".encode()
        ).hexdigest()
        conn.execute(
            """
            INSERT INTO signals (
                id, company_id, event_id, signal_type, title, description,
                direction, severity, evidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (sid, company_id, event_id, s["signal_key"], s["title"], s["description"],
             s["direction"], s["severity"],
             json.dumps({
                 "metric_keys": s["metric_keys"],
                 "fact_keys": s["fact_keys"],
                 "trigger_values": s["trigger_values"],
                 "rule_text": s["rule_text"],
                 "category": s["category"],
                 "segment": s.get("segment"),
                 "geography": s.get("geography"),
                 "catalog": "investor_presentation/presentation_signals.json",
             })),
        )
    conn.commit()

print(f"Evaluated {len(signals_catalog)} presentation signal rules -> {len(fired_signals)} fired\n")
for s in fired_signals:
    dims = []
    if s.get("segment"):
        dims.append(f"segment={s['segment']}")
    if s.get("geography"):
        dims.append(f"geography={s['geography']}")
    dim_note = f" [{', '.join(dims)}]" if dims else ""
    print(f"  [{s['severity']}/{s['direction']}] {s['title']}{dim_note}")
    print(f"        rule: {s['rule_text']}  ->  {s['trigger_values']}")


Evaluated 34 presentation signal rules -> 1 fired

  [MEDIUM/NEGATIVE] Potential Inventory Build [segment=ATF, geography=Domestic India]
        rule: production_to_sales_ratio > 1.1  ->  {'production_to_sales_ratio': 8.16}


## STEP 7 / 7  -  ALERTS

In [16]:
# =============================================================================
# STEP 7 / 7  -  ALERTS
# Read the fired signals back from the DB and present them as alerts,
# then print a final per-event DB summary.
# =============================================================================
with db_connect() as conn:
    alert_rows = conn.execute(
        """
        SELECT signal_type, title, description, direction, severity, evidence
        FROM signals WHERE company_id = ? AND event_id = ?
        """,
        (company_id, event_id),
    ).fetchall()

    counts = {
        "extracted_values": conn.execute(
            "SELECT COUNT(*) c FROM extracted_values WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "metric_values": conn.execute(
            "SELECT COUNT(*) c FROM metric_values WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "signals": conn.execute(
            "SELECT COUNT(*) c FROM signals WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
    }

_SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}

print("=" * 70)
print(f"ALERTS  -  {SYMBOL}  -  {EVENT_TYPE}  -  {PERIOD_LABEL}")
print("=" * 70)

if not alert_rows:
    print("\nNo signals triggered for this filing.")
else:
    ordered = sorted(alert_rows, key=lambda r: _SEVERITY_ORDER.get(r["severity"], 9))
    for r in ordered:
        evidence = json.loads(r["evidence"]) if r["evidence"] else {}
        triggers = evidence.get("trigger_values", {})
        print(f"\n  [{r['severity']}] {r['title']}  ({r['direction']})")
        print(f"      {r['description']}")
        if triggers:
            trig = ", ".join(f"{k}={v}" for k, v in triggers.items())
            print(f"      triggered by: {trig}")

print("\n" + "-" * 70)
print(f"DB summary for event {event_id[:12]}...:")
print(f"  extracted_values : {counts['extracted_values']}")
print(f"  metric_values    : {counts['metric_values']}")
print(f"  signals          : {counts['signals']}")
print(f"\nAll seven steps complete. Data persisted to {DB_PATH}")

ALERTS  -  RELIANCE  -  Investor Presentation  -  Q4 FY2025-26

  [MEDIUM] Potential Inventory Build  (NEGATIVE)
      Production materially exceeds sales volume, suggesting possible inventory accumulation.
      triggered by: production_to_sales_ratio=8.16

----------------------------------------------------------------------
DB summary for event 2fc87238a563...:
  extracted_values : 71
  metric_values    : 20
  signals          : 1

All seven steps complete. Data persisted to /Users/prairit/capital-nerve/v3/data/capital_nerve.db
